<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 9 - Land Use & Buildings

#### 3D Urban Visualization and Data Aggregation (San Francisco)

### CP101: Introduction to Urban Data Analytics

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB09.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

By the end of this session, you will be able to:

1. Load Overture Maps building footprints saved from a bounding-box query and confirm their CRS.
2. Clean a real-world GeoDataFrame by dropping sparse columns with `dropna(thresh=...)` and filling missing height values.
3. Normalize heavy-tailed height data on a log scale and apply a continuous color map for a 3D building visualization.
4. Attach zoning to buildings with a spatial join, catch the duplicate rows an `intersects` join creates, and give each building the zone that covers the largest share of its footprint.
5. Compute multi-metric group statistics (median, max) by zoning category using a reusable Split-Apply-Combine function.

---

In this lesson, we use **San Francisco building footprints** and **zoning districts** to analyze urban density and visualize the city in 3D.

### **The story**

Zoning laws dictate how tall buildings can be and what they can be used for, directly shaping the skyline of a city.

We will investigate San Francisco's built environment:

1. **What are the typical and maximum building heights across different zoning districts?**
2. **How do we efficiently compute multi-metric profiles for massive urban datasets?**
3. **How can we visualize hundreds of thousands of buildings in an interactive 3D web map?**

---

### Data layers used in this notebook

- **San Francisco Building Footprints** (includes height attributes)
- **San Francisco Zoning Districts**

Load SF zoning polygons, retrieve building footprints from Overture Maps, compute a vectorized spatial join (buildings + zoning), and visualize both layers using **Lonboard**. You can access the documentation, providing metadata about building dataset from [Official Overture Maps Schema Documentation](https://docs.overturemaps.org/schema/reference/buildings/building/).

First install the required packages:

In [ ]:
!pip -q install "geopandas[all]" pyproj lonboard mapclassify ipywidgets overturemaps palettable


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# SF bounding box
bbox = -122.5145, 37.7085, -122.3716, 37.8092

Overture only keeps each release for about two months; once a release ages out, listing that path returns an error.
To avoid that, we can find the latest release using the below code, and use it to fetch the data.

This is the way to get the data directly from Overture Maps, but to not overflow the server, we will just use the data we already have in `data/raw`.
The file `SF_Overture_Buildings_20261016.parquet` was saved with this exact code and the same `bbox`, using release `2026-09-23.1`.
The code is shown here for reference only, so please do not run it on the hub.
To run it on your own computer, install the `overturemaps` package first.

```python
import overturemaps
import requests

# Overture publishes a STAC catalog that lists every release it still hosts.
# The catalog is occasionally unavailable, so we keep a known-good release as a
# fallback: update PINNED_RELEASE if the fetch below ever falls back on you.
PINNED_RELEASE = "2026-09-23.1"

try:
    response = requests.get("https://stac.overturemaps.org/catalog.json", timeout=30)
    response.raise_for_status()
    latest_release = response.json()["latest"]
    print(f"The latest release is: {latest_release}")
except Exception as err:
    latest_release = PINNED_RELEASE
    print(f"Could not read the release catalog ({err}).")
    print(f"Falling back to the pinned release: {latest_release}")

# Pulls the newest release directly into a GeoPandas GeoDataFrame! This can take a while!
sfgdf = (
    overturemaps
    .core
    .geodataframe(
      "building",
      bbox=bbox,
      release = latest_release,
      )
)
```

In [ ]:
# Overture buildings for the same bbox, saved from release 2026-09-23.1
SF_BUILDINGS = './data/raw/SF_Overture_Buildings_20261016.parquet'

sfgdf = gpd.read_parquet(SF_BUILDINGS)

In [ ]:
sfgdf.head(3)

In [ ]:
# Overture data are in EPSG:4326, so set it only if the file has no crs
if sfgdf.crs is None:
    sfgdf = sfgdf.set_crs(4326)

sfgdf.crs

In [ ]:
sfgdf.info()

In [ ]:
# Compute total non-NaNs per column
valid_counts = sfgdf.count()
valid_percentages = (valid_counts / len(sfgdf)) * 100


In [ ]:
summary_df = pd.DataFrame({
    'Non-NaN Count': valid_counts,
    'Valid %': valid_percentages
})

print("******* Column Summary *******")
summary_df

### ✏️ Try it Out - How many buildings have a height?

The 3D map needs a height for every building, so first check how many have one.
Store the **number** of buildings with a missing `height` in `n_missing_height`.
Store the **percentage** of buildings that do have a height in `pct_with_height`, as a number between 0 and 100.

In [ ]:
n_missing_height = ...
pct_with_height = ...

print("Buildings with no height:", n_missing_height)
print("Share with a height:", round(pct_with_height, 2), "%")

In [ ]:
grader.check("q1_height_coverage")

### The Automatic Drop (Using thresh)

Once you decide on your minimum acceptable number of valid rows, you can use the `dropna()` method with the thresh parameter.

The `thresh=N` argument tells pandas: "Keep this column ONLY if it has at least N non-NaN values."

In [ ]:
# I want only columns that have more than 30% non-nan data for visualization purpose
# thresh counts rows, so we take 30% of however many buildings came back
thresh = len(sfgdf) * 0.3

cleaned_sfgdf = (
    sfgdf
    .dropna(
      axis=1,
      thresh=thresh
      )
)

print("\n--- After Dropping ---")
cleaned_sfgdf.columns.tolist()


### Prepping the Heights

We need to tell our 3D map exactly how tall to draw each building. Real-world data is messy, and some buildings don't have a height recorded. If we pass a missing value to the map's elevation parameter, it will error out.

In [ ]:
heights = (
    cleaned_sfgdf["height"]
    .fillna(1)
    .to_numpy()
)


What this does:

- `sfgdf["height"]`: selects the column that contains building heights.
- `.fillna(1)`: fills missing values with 1 meter. Buildings with unknown height will show up as flat, pancake-like footprints instead of disappearing or crashing the map.
- `.to_numpy()`: converts the cleaned column to a NumPy array, which the 3D mapping engine expects instead of a pandas Series.

### The Math (Coloring the Skyscrapers)

Building heights in San Francisco are heavy-tailed: many short buildings and only a handful of very tall skyscrapers. A linear color scale would wash almost all buildings into the same pale color, leaving only the tallest towers visible. We use a Logarithmic Normalizer (`LogNorm`) instead, so shorter buildings stay visually distinguishable while the tall outliers get compressed into the scale rather than dominating it.


In [ ]:
from matplotlib.colors import LogNorm

normalizer = LogNorm(
    1,
    heights.max(),
    clip=True
    )
normalized_heights = normalizer(heights)

- `1` and `heights.max()`: the boundaries of the curve: the minimum (our filled-in placeholder value) and the tallest building in the city.
- `clip=True`: if a data glitch produces a value outside the boundaries (e.g. a negative height), this clips it back into range instead of erroring.
- `normalized_heights = normalizer(heights)`: applies the curve to the raw heights, producing a mathematically balanced score for each building, ready to map onto a color scale.

### Painting the Map (The Color Palette)

We have our building heights, and we have smoothed them out with math. Now, we need to actually paint them.


Before we can color the map, we need a color palette. This line hands us a ready-to-use orange gradient instead of manually specifying 9 hex codes.

In [ ]:
from palettable.colorbrewer.sequential import Oranges_9

Oranges_9.mpl_colormap

- `Oranges_9`: a pre-made palette of 9 shades of orange.

In [ ]:
from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap

colors = apply_continuous_cmap(normalized_heights, Oranges_9)

- `apply_continuous_cmap`: maps our normalized heights onto the orange palette, producing a color for every building. A short building gets a light peach color, a medium building gets solid orange, and the tallest towers get dark burnt orange.

### Building the 3D City from a GeoDataFrame
This is where our data, heights, and colors come together as 3D shapes.

Massive spatial datasets are heavy, and traditional plotting libraries struggle to render them without freezing. Lonboard sends the data directly to your computer's GPU, so it can render hundreds of thousands of 3D buildings smoothly in the browser.

In [ ]:
from lonboard import Map, PolygonLayer, SolidPolygonLayer
from lonboard.basemap import MaplibreBasemap, CartoStyle

from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap
from lonboard import viz

In [ ]:
layer = (
    PolygonLayer
    .from_geopandas(
      cleaned_sfgdf[["id", "height", "geometry"]], # We subset columns so the map loads faster!
      extruded=True,
      get_elevation=heights,
      get_fill_color=colors,
  )
)

- `.from_geopandas(...)`: Lonboard converts our GeoDataFrame into its own fast binary format under the hood.
- `cleaned_sfgdf[["id", "height", "geometry"]]`: we subset to only the columns the map needs, so it loads faster.
- `extruded=True`: tells the map to pull each flat building footprint upward into a 3D block, using the elevation values, instead of drawing flat shapes.
- `get_elevation` & `get_fill_color`: pass in the cleaned heights (how tall to draw each block) and the color mapping (what shade to fill it).

### Setting the Stage
We built our 3D layer, but it needs a camera position and a basemap to render against.

In [ ]:
view_state = {
    "longitude": -122.4194,  # SF center
    "latitude":  37.7749,
    "zoom": 12,
}

m = Map(
    layer,
    view_state=view_state,
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatter),  # dark basemap for 3D
)
m

- `view_state`: sets the camera's starting position: the GPS coordinates for the center of San Francisco, zoomed out enough (`zoom: 12`) for the whole city to fit on screen.
- `Map(...)`: combines the 3D layer and the view state into one map object (`m`).
- `CartoStyle.DarkMatter`: a dark basemap. Extruded buildings read best against a dark background, and the orange gradient stands out without competing with water and parks.
- `m`: typing the variable name by itself renders the interactive 3D map in the notebook.

### Adding Zoning Map

Height and density are rarely accidental; they are the physical manifestation of municipal policy. To understand the spatial distribution of the San Francisco skyline, we must intersect our building footprints with the city’s official Zoning Map. This dataset serves as the legal backbone for land-use intensity, setback requirements, and height limits.

In [ ]:

SF_ZONING = './data/raw/SF_Zoning_Map_20251109.parquet'

zoning = gpd.read_parquet(SF_ZONING)

if zoning.crs is None:
    zoning = zoning.set_crs(4326)


In [ ]:
zoning.columns

In [ ]:
# Subset to essential attributes: zoning class, district name, and generalized category
zoning_filtered = zoning[['zoning_sim',
                          'districtna',
                          'url',
                          'gen',
                          'zoning',
                          'codesectio',
                          'geometry']].copy()

len(zoning_filtered.zoning.unique()) #how many unique zoning do we have?

In [ ]:
len(zoning_filtered.gen.unique())

In [ ]:
zoning_filtered.sort_values(
    by='zoning',
    inplace=True
    )

In [ ]:
zoning_filtered.sample(3)

In [ ]:
zoning_filtered.gen.value_counts()

### Spatial Join - Intersecting Buildings with Zoning

To attach zoning to each building, we perform a spatial join (`sjoin`).
Unlike a standard tabular join on shared IDs, this operation uses a spatial predicate (`intersects`) to decide which rows match.
Every building footprint in `sfgdf` is tested against the zoning polygons, and wherever they overlap, the building inherits that district's attributes, such as its generalized category (`gen`) and zoning code.
Keep an eye on the row count, because a building can overlap more than one polygon.

In [ ]:
if not sfgdf.crs == zoning_filtered.crs:
  sfgdf = sfgdf.to_crs(zoning_filtered.crs)

In [ ]:
# A left spatial join keeps every building, matched to every zoning polygon it touches
buildings_zoned_intersects = gpd.sjoin(
    sfgdf,
    zoning_filtered,
    how="left",
    predicate="intersects",
).drop(columns=["index_right"])

### One building, one row?

`sfgdf` has one row per building, so a left join should return one row per building too.
It does not.
A building that touches two zoning polygons matches both of them, and the join returns it twice, once with each zone.
Every statistic and map below would then count that building twice.

### ✏️ Try it Out - Count the extra rows

Each building has a unique `id`.
Store the number of **distinct** buildings in `buildings_zoned_intersects` in `n_buildings`.
Store the number of **extra** rows the join created in `n_extra_rows`.

In [ ]:
n_buildings = ...
n_extra_rows = ...

print("Rows after the join:", len(buildings_zoned_intersects))
print("Distinct buildings:", n_buildings)
print("Extra rows:", n_extra_rows)

In [ ]:
grader.check("q2_join_duplicates")

### Which zone does a split building belong to?

San Francisco's Planning Code does not give a building a single zone.
For housing density, it applies each district's limit to the portion of the lot that lies inside that district (Planning Code §207(b)(5)).
Most district boundaries follow lot lines (§105(b)), so many of the extra rows are buildings that only touch a boundary, or cross it by a sliver because Overture's footprints and the City's lot lines do not line up exactly.
Some buildings really do sit across two districts: about 2,400 have more than a tenth of their footprint in a second district.

### ✏️ Try it Out - Why the extra rows?

In two or three sentences, explain why the `intersects` join returns more rows than there are buildings.
Then explain why giving each building **one** zone is a choice we make for our analysis rather than what the Planning Code says, and say which buildings that choice is most likely to describe wrongly.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### One zone per building - the largest-share rule

We give each building the zone that covers the **largest share of its footprint**.
For a building that only grazes a boundary, that is the zone it actually sits in.
For a building that really crosses a boundary, it is the zone that governs most of it.
Assigning the zone that contains the building's centroid, as in Lab 7, gives the same answer for all but 37 buildings here, but the largest-share rule also works for L-shaped buildings, whose centroid can fall outside the footprint.

The rule takes four steps, all with tools you have used before:

1. Reproject both layers to a CRS measured in feet (EPSG:2227, California State Plane zone 3), because we are about to compare areas. The next cell does this for you.
2. `gpd.overlay(..., how="intersection")` cuts each building into pieces, one per zoning district it overlaps (Lab 7).
3. Measure each piece's area in square feet.
4. Sort the pieces with the largest first, and keep only the first row for each building `id`.

In [ ]:
buildings_reproj_2227 = sfgdf[["id", "geometry"]].to_crs(2227)
zoning_reproj_2227 = zoning_filtered.to_crs(2227)

### ✏️ Try it Out - Apply the rule

Fill in the blanks to build `pieces` (the overlay), its `piece_sqft` column, and `largest_piece` (one row per building, the largest piece).
The rest of the cell attaches each building's chosen zone back onto `sfgdf` as `buildings_zoned`.
The overlay takes about half a minute.

In [ ]:
pieces = gpd.overlay(
    ...,
    ...,
    how=...,
)

pieces["piece_sqft"] = ...

largest_piece = (
    pieces
    .sort_values(...)
    .drop_duplicates(...)
)

# Attach the chosen zone to the original buildings, which stay in EPSG:4326 for mapping
zone_columns = ["id", "zoning_sim", "districtna", "url", "gen", "zoning", "codesectio"]

buildings_zoned = sfgdf.merge(
    largest_piece[zone_columns],
    on="id",
    how="left",
)

print("Buildings:", len(sfgdf))
print("Rows after the largest-share rule:", len(buildings_zoned))

In [ ]:
grader.check("q4_largest_share")

### Analyzing Urban Morphology (Height by Zone)

To understand the urban morphology of San Francisco, we look at the physical dimensions of buildings within each zoning category.
The median height of the buildings in each category shows where the city is built tall and where it is built low.
The `summarize_stats` function performs a multi-metric Split-Apply-Combine operation, so you can calculate several statistics at once.
You can copy it to a `.py` file, save it in your project, and import it across your notebooks in the final project.
You will reuse it later in this lab for building footprints.

In [ ]:
def summarize_stats(
    gdf: gpd.GeoDataFrame,
    group_col: str,
    value_col: str,
    agg_funcs: list = ["median", "max"]
) -> pd.DataFrame:
    """
    Enhanced aggregator to calculate multiple statistics for a specific
    attribute across spatial categories in a single pass.
    """
    # group and aggregate
    out = (
        gdf.groupby(group_col)[value_col]
        .agg(agg_funcs)
        .reset_index()
    )

    # Flatten multi-index columns and rename for clarity
    # e.g., 'median' becomes 'median_height'
    agg_col_names = []
    for f in agg_funcs:
        agg_col_names.append(f"{f}_{value_col}")
    out.columns = [group_col] + agg_col_names

    return out.round(2).sort_values(f"{agg_funcs[0]}_{value_col}", ascending=False)


The `summarize_stats` function is designed to perform a multi-metric **Split-Apply-Combine** operation:

- `group_col` (Split): This defines the categorical "buckets" for the analysis, such as splitting the city into its respective `zoning` districts.

- `value_col` (Apply): This is the numerical attribute to be measured across those groups, such as `height` or `num_floors`.

- `agg_funcs` (Combine): This parameter accepts a **list** of strings (e.g., `["median", "max"]`). This allows the function to compute several statistics in a single pass.

- Dynamic Renaming: The function automatically renames the resulting columns (e.g., `median_height`, `max_height`) so the output is "tidy" and ready for analysis.

We prefer the Median over the Mean here because building height data is notoriously skewed (Remember what we discussed in the class!). While the Mean can be distorted by a single iconic skyscraper, the Median identifies the "typical" structure, providing a more accurate reflection of the neighborhood's built character. For planning purposes, comparing the Median (typical height) against the Maximum (tallest structure) reveals how much "height variety" exists within a specific zoning envelope.

Applying this function to `buildings_zoned` turns 161,848 individual buildings into one row per zoning category.
Note what the table cannot show.
The zoning layer records each district's category, not its legal height limit.
San Francisco sets height limits in a separate Height and Bulk Districts map, so comparing built heights with permitted heights would need that layer too.

In [ ]:
# Analyze the median and max building height across broader generalized ('gen') zoning categories
zoning_profile = summarize_stats(
    buildings_zoned,
    group_col="gen",
    value_col="height",
    agg_funcs=["median", "max"]
)
zoning_profile

A critical step in any spatial workflow is the verification of units against the source metadata. While San Francisco's municipal codes are historically defined in feet, the Official Overture Maps Documentation (linked at the top of this notebook) explicitly defines the height attribute in meters.

### ✏️ Try it Out - Heights in feet

Add two columns to `zoning_profile`, `median_height_ft` and `max_height_ft`, by converting the meter values to feet ($1$ meter $\approx 3.28$ feet).

In [ ]:
zoning_profile["median_height_ft"] = ...
zoning_profile["max_height_ft"] = ...

zoning_profile

In [ ]:
grader.check("q5_height_ft")

### ✏️ Try it Out - How big are the buildings in each zone?

Height is one dimension of a building, and its footprint is another.
Add a column `footprint_sqft` to `buildings_zoned` with each building's footprint area in **square feet**.
`buildings_zoned` is in EPSG:4326, so measure the area on a reprojected copy, `buildings_zoned_reproj_2227`.
Then use `summarize_stats` to build `footprint_profile`, the median and maximum footprint for each `gen` category.

In [ ]:
buildings_zoned_reproj_2227 = ...
buildings_zoned["footprint_sqft"] = ...

footprint_profile = summarize_stats(
    ...
)

footprint_profile

In [ ]:
grader.check("q6_footprint_area")

### ✏️ Try it Out - Read the two profiles

Use `zoning_profile` and `footprint_profile` to answer in three or four sentences.

1. For one category, compare the median and the maximum height. What does the gap tell you about how heights are distributed in that category?
2. With the duplicated join, Public's tallest building was 244.4 m. With the largest-share rule, it is 122 m. Using your answer about split buildings, explain what happened.
3. Which categories have the largest typical footprints, and does that fit how those uses are built?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Mapping 3D Buildings with Zoning

### Step 1 - Configure the basemap

A 3D map reads best on a dark basemap, so we use CARTO's Dark Matter style.
We try the newer `MaplibreBasemap` API first, and if it is not available, we fall back to the classic `basemap_style` URL.

In [ ]:
try:
    from lonboard.basemap import MaplibreBasemap, CartoStyle
    map_kwargs = {"basemap": MaplibreBasemap(style=CartoStyle.DarkMatter)}
except (ImportError, AttributeError):
    map_kwargs = {"basemap_style": "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json"}


### Step 2 - Prepare the color scheme for zoning categories

Assign a distinct, colorblind-friendly color (CARTO `Safe_10`) to each zoning group.
The map uses a dark basemap, so we keep only the six `Safe_10` colors light enough to read on it.
Categories get their colors in alphabetical order, and *Unknown* is always gray, so the legend below stays correct if the data changes.

In [ ]:
from palettable.cartocolors.qualitative import Safe_10

# Extract zoning categories and handle missing values
zoning_categories = (
    buildings_zoned["gen"]
    .astype("string")
    .fillna("Unknown")
)

# Named categories in alphabetical order, so each one keeps its color if the data changes
named_categories = sorted(
    zoning_categories[zoning_categories != "Unknown"].unique()
)

# Convert palette colors from [0,1] floats to [0,255] uint8 RGB
palette_colors = (np.array(Safe_10.mpl_colors) * 255).astype(np.uint8)

# Keep the six Safe_10 colors light enough to read on a dark basemap
light_colors = palette_colors[[0, 1, 2, 5, 6, 7]]

# Guard against more categories than available colors
if len(named_categories) > len(light_colors):
    raise ValueError(
        f"We have {len(named_categories)} zoning categories but only "
        f"{len(light_colors)} light colors. "
        f"Consider using a larger palette or grouping some categories."
    )

# Category → RGB mapping, with Unknown always gray
color_mapping = {}
for i, category in enumerate(named_categories):
    color_mapping[category] = light_colors[i].tolist()
color_mapping["Unknown"] = [136, 136, 136]

unique_categories = pd.Index(named_categories + ["Unknown"])

In [ ]:
color_mapping

### Step 3 - Apply colors to buildings by zoning category

Convert to a categorical dtype with a fixed category ordering and apply the mapping
to vectorize color assignment. Use semi-transparency so overlaps remain legible.


In [ ]:
zoning_categorical = zoning_categories.astype(pd.CategoricalDtype(categories=unique_categories))
building_colors = apply_categorical_cmap(zoning_categorical, color_mapping, alpha=160)


### Step 4 - Prepare building heights for 3D extrusion

Clean height values, default missing to 1.0, and clamp to a minimum of 1.0.


In [ ]:
building_heights = np.asarray(buildings_zoned["height"], dtype=float)
building_heights = np.nan_to_num(building_heights, nan=1.0)
building_heights = np.maximum(building_heights, 1.0)


### Step 5 - Create the 3D buildings layer

Use `SolidPolygonLayer` with extrusion, zoning-based fill colors, and subtle outlines.


In [ ]:
buildings_layer = SolidPolygonLayer.from_geopandas(
    buildings_zoned,
    extruded=True,                  # enable 3D extrusion
    get_elevation=building_heights, # cleaned heights
    elevation_scale=1.0,            # 1:1 scale
    get_fill_color=building_colors, # color by zoning
    get_line_color=[30, 30, 30, 60],
    wireframe=False,
)
layers = [buildings_layer]


### Legend - Zoning Categories

Buildings are colored by their generalized zoning category (`gen`), using the light colors of the CARTO `Safe_10` colorblind-friendly palette, assigned in alphabetical order:

- **Commercial** - light blue (`#88CCEE`)
- **Industrial** - rose (`#CC6677`)
- **Mixed** - sand (`#DDCC77`)
- **Mixed Use** - purple (`#AA4499`)
- **Public** - teal (`#44AA99`)
- **Residential** - olive (`#999933`)
- **Unknown** - gray (`#888888`), the 14 buildings that overlap no zoning polygon

**Mixed** and **Mixed Use** are different categories in the zoning data.
Mixed is two special districts, Pier 70 and the Potrero Power Station, while Mixed Use covers the citywide neighborhood commercial and mixed-use districts.

### Step 6 - Create the map and set the initial view

Initialize the interactive map and fly to downtown San Francisco.
To tilt the view, drag with right-click (or Ctrl+drag) once the map appears.


In [ ]:
m = Map(layers=layers, height=700, **map_kwargs)

m.set_view_state(
    longitude=-122.419,
    latitude=37.77537,
    zoom=14,
)

m


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Optional Extension (Bonus, Not Required)

This is a second, larger mini-project meant to be done asynchronously, after class, at your own pace, so come back to it when you have time. It is not required in-session work.

So far we have been working with zoning data. SF's Planning Department publishes a richer classification called land use, which distinguishes between residential, office (MIPS), retail, industrial (PDR), civic/institutional (CIE), medical, and hotel uses on a parcel-by-parcel basis.

Load the land use data with:

```python
  SF_LU = './data/raw/SF_LU_2023_20260330.parquet'
  lu = gpd.read_parquet(SF_LU)
```

Give each building one land use category (`landuse`), using the largest-share rule from the zoning section.
Group by `landuse` and compute the minimum, maximum, and median `footprint_sqft` for each category.
Sort by median footprint area, largest first, and display the result.

What do you notice? Which land use categories tend to have the largest building footprints? Does the pattern make sense given what you know about how these uses are built?

Use lonboard to create visualizations for your analysis (you can use the same 3D height and overlay it or color it by different values.)

Hints:
- You will have more than 10 categories of land use, so you will need a different color map. Check [this link](https://jiffyclub.github.io/palettable/) for a color map you like and replace Safe_10 with that.

- Lonboard renders through your browser's WebGL/GPU pipeline (via deck.gl) rather than the CPU-bound rendering matplotlib uses. Pushing tens of thousands of building footprints with 3D extrusion is exactly the kind of workload GPUs handle well, and this land-use dataset is heavier than the zoning example. If the map runs slowly and you're on Colab, switch your runtime to use a GPU (Runtime > Change runtime type > choose a GPU type).

**Publish it:** once your land-use map looks right, export it with `.to_html(...)` and publish it to your GitHub Pages repo. We cover exporting and publishing in **Lab 12 - Public-Facing Story**; if you are doing this bonus before then, that notebook has the steps. Submit the live URL.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary - 3D Building Visualization and Zoning Analysis

| Task | Pattern |
|---|---|
| Load saved building footprints | `gpd.read_parquet(path)` |
| Fetch building footprints for a bounding box (reference) | `overturemaps.core.geodataframe("building", bbox=..., release=...)` |
| Fill and normalize skewed height data | `.fillna(1)` then `LogNorm(1, heights.max())` |
| Apply a continuous color map | `apply_continuous_cmap(normalized_heights, Oranges_9)` |
| Attach zoning attributes to buildings | `gpd.sjoin(buildings, zoning, how="left", predicate="intersects")` |
| Count the rows a join duplicated | `len(joined) - joined["id"].nunique()` |
| Give each building the zone covering most of it | `gpd.overlay(..., how="intersection")`, `.area`, `.sort_values(..., ascending=False)`, `.drop_duplicates(subset="id")` |
| Measure footprint area in square feet | `gdf.to_crs(2227).area` |
| Summarize a metric by category | `summarize_stats(gdf, group_col=..., value_col=..., agg_funcs=["median", "max"])` |
| Assign categorical colors | `apply_categorical_cmap(categorical_series, color_mapping, alpha=...)` |
| Render an extruded 3D map | `SolidPolygonLayer.from_geopandas(..., extruded=True, get_elevation=..., get_fill_color=...)` |

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Key Takeaways

1. **Query Overture once, then work from a saved copy** - a bounding-box query fetches only the buildings you need, and saving the result keeps a whole class from hitting the server at the same time.
2. **Real-world spatial data needs cleaning before it can drive a 3D map** - dropping sparse columns and filling missing heights with an obvious placeholder value keeps the pipeline from erroring out or silently misleading.
3. **A log-scale color normalization keeps heavy-tailed data readable** - without it, a handful of skyscrapers would wash out every other building into the same color.
4. **Check the row count after every spatial join** - an `intersects` join matches a building to every zone it touches, so 161,848 buildings became 170,351 rows, and when you need one zone per building you have to state a rule, such as the zone covering the largest share of the footprint.
5. **Median, not mean, describes a "typical" building when height data is skewed** - comparing median against max reveals how much height variety exists within a zoning category.
6. **In the bonus, exporting to HTML and hosting on GitHub Pages turns a notebook-only map into a shareable, standalone web page.**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Upload your submission zip to **LAB09 - Land Use & Buildings** on [Gradescope](https://www.gradescope.com/).

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)